# 10年定着予測 - TabPFN v2 をアンサンブル相手にする（63_）

## 位置づけ

第88節のEDAで「伸びしろは特徴量ではなくモデリング・アンサンブル側」と結論した。その一手として、
HuggingFaceの表形式基盤モデル **TabPFN v2**（[Prior-Labs/TabPFN-v2-clf](https://huggingface.co/Prior-Labs/TabPFN-v2-clf)）
をアンサンブル相手として検証する。

## なぜ TabPFN v2 か

Transformerが**in-context learningで1回のforward passで**表形式予測を解く、GBDTとは推論機構が
根本的に違うモデル。合成タスクのみで事前学習されており、勾配ブースティングとは全く別の
帰納バイアスを持つ ＝ **アンサンブルの多様性として理想的**。

データとの相性も良い:

| 項目 | TabPFN v2 | 本データ |
|---|---|---|
| サンプル数 | 小規模向け（〜10,000） | 2,761行 ✓ |
| 特徴量数 | 〜500 | 441列 ✓ |
| 欠損値 | **そのまま扱える** | NaN多数 ✓ |
| 前処理 | スケーリング・one-hot**不要** | ✓ |

**バージョンは v2 を使う。** 新しい TabPFN 2.5/2.6/3 は非商用ライセンスでコンペ利用がグレーだが、
**v2 は Apache 2.0（帰属表示が追加要件）** で安全側のため。**GPUランタイムが必要**
（CPUではサンプル数制限に引っかかる）。

## 事前登録: 合格基準

`55_`のGRUは相関0.31-0.35という本物の多様性がありながら、**絶対性能の差が0.136あったせいで
どの重みでもブレンドが改善しなかった**（[[ensemble-oof-overfitting]] addendum 6）。
同じ失敗を繰り返さないため、実行前に基準を決めておく:

- **足切り**: TabPFN単体のvalが、同一split・同一特徴量のCatBoost比 **+0.02より悪ければ不合格**
  （ブレンドしても希釈するだけなので、そこで打ち切る）
- **採用検討**: 合格した場合のみブレンド重みを走査する。ただし最終判断はPublicのみ
  （[[validation-asymmetry]]）
- 重みは**固定値の走査**にとどめ、学習された重みは使わない（[[ensemble-oof-overfitting]]）

## 検証する構成

TabPFNは特徴量数が多いと性能が落ちるとされるため、列数の感度も同時に見る。

| 構成 | 列数 |
|---|---|
| full441 | 441（現行パイプラインそのまま）|
| top150 | CatBoostの重要度上位150列 |

---

## ライセンス表記（Prior Labs License 第10条）

> **Built with PriorLabs-TabPFN**

`tabpfn==2.2.1`（v2世代）は **Prior Labs License v1.1**（Apache 2.0派生、帰属表示要件つき）で
**商用利用可**。SIGNATE参加規約 第2条8項（商業利用が禁止されているOSSの利用禁止）に適合する。
**2.5/2.6/3系は非商用ライセンスなので使用不可。** 本ノートブックはバージョンを固定し実行時に検証する。


In [1]:
# 重いライブラリは最初にまとめて入れる。
# tabpfn は後半で入れるとランタイム再起動時に特徴量生成(約15分)をやり直す羽目になるため、ここで入れる。
#
# ⚠️ tabpfn はバージョン固定が必須。素の `pip install tabpfn` は最新(8.x)を入れてしまい、
#    そちらは Prior Labs のライセンス同意＋APIトークンが必要（TabPFNLicenseError で落ちる）。
#    2.x 系が TabPFN v2 世代で、Apache 2.0（帰属表示が追加要件）・ライセンスゲート無し。
!pip install -q catboost optuna "tabpfn==2.2.1"

# 出る「ERROR: pip's dependency resolver...」は依存解決の警告であって失敗ではない。
#   - huggingface-hub 0.36.x への降格は tabpfn 2.2.1 の要求(`huggingface-hub<1`)通りで正しい
#   - gradio / transformers / google-colab の警告は本ノートブックで使わないので無視してよい


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.7/173.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.3/504.3 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 6.2

In [2]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [3]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))


Mounted at /content/drive


In [4]:
import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


In [5]:
SCRIPT_NAME = "63_tabpfn_ensemble"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

RESET_CHECKPOINT = True  # Trueにすると既存チェックポイントを削除して最初から再計算する
if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("チェックポイントを削除しました（全構成を再計算します）")

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")


[2026-08-16 01:26:14] [INFO] === [63_tabpfn_ensemble] 実験開始 ===


INFO:63_tabpfn_ensemble:=== [63_tabpfn_ensemble] 実験開始 ===


[2026-08-16 01:26:15] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260816


INFO:63_tabpfn_ensemble:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260816


[2026-08-16 01:26:15] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/63_tabpfn_ensemble_checkpoint.csv


INFO:63_tabpfn_ensemble:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/63_tabpfn_ensemble_checkpoint.csv


[2026-08-16 01:26:15] [INFO] チェックポイントは未作成（新規実行）


INFO:63_tabpfn_ensemble:チェックポイントは未作成（新規実行）


In [6]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")


[2026-08-16 01:26:18] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:63_tabpfn_ensemble:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-16 01:26:18] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:63_tabpfn_ensemble:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-16 01:26:18] [INFO] 定着率: 0.5647


INFO:63_tabpfn_ensemble:定着率: 0.5647


[2026-08-16 01:26:18] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:63_tabpfn_ensemble:Train IDs: 2761, Test IDs: 2502


## 0. 早期退職者の特定

`月末在籍状態 == "退職"` の行を持つ社員が、0-23ヶ月の観測期間中に退職した社員。
Train に129名（全員ラベル0）、Test には0名（[[test-set-is-survivor-filtered]]）。


In [7]:
EARLY_LEAVER_IDS = set(train_monthly.loc[train_monthly["月末在籍状態"] == "退職", ID_COL].unique())
_test_early = set(test_monthly.loc[test_monthly["月末在籍状態"] == "退職", ID_COL].unique())

logger.info(f"Train 早期退職者: {len(EARLY_LEAVER_IDS)}名 / {len(train_ids)}名 ({len(EARLY_LEAVER_IDS)/len(train_ids):.1%})")
logger.info(f"Test  早期退職者: {len(_test_early)}名 / {len(test_ids)}名")
_y_idx = train_persona.set_index(ID_COL)[TARGET_COL]
logger.info(f"早期退職者のラベル平均: {_y_idx.loc[list(EARLY_LEAVER_IDS)].mean():.4f}（0.0のはず）")
logger.info(f"定着率: 全体 {y_train.mean():.4f} / 早期退職者を除く {_y_idx[~_y_idx.index.isin(EARLY_LEAVER_IDS)].mean():.4f}")
assert len(_test_early) == 0, "Testに早期退職者が存在する。前提が崩れているので調査すること"


[2026-08-16 01:26:18] [INFO] Train 早期退職者: 129名 / 2761名 (4.7%)


INFO:63_tabpfn_ensemble:Train 早期退職者: 129名 / 2761名 (4.7%)


[2026-08-16 01:26:18] [INFO] Test  早期退職者: 0名 / 2502名


INFO:63_tabpfn_ensemble:Test  早期退職者: 0名 / 2502名


[2026-08-16 01:26:18] [INFO] 早期退職者のラベル平均: 0.0000（0.0のはず）


INFO:63_tabpfn_ensemble:早期退職者のラベル平均: 0.0000（0.0のはず）


[2026-08-16 01:26:18] [INFO] 定着率: 全体 0.5647 / 早期退職者を除く 0.5923


INFO:63_tabpfn_ensemble:定着率: 全体 0.5647 / 早期退職者を除く 0.5923


## 1. 基本特徴量関数の定義（split非依存、`51_`と同一ロジック）

In [8]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")


✅ split非依存の基本特徴量関数定義完了


In [9]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")


[2026-08-16 01:26:18] [INFO] ------------------------------------------------------------


INFO:63_tabpfn_ensemble:------------------------------------------------------------


[2026-08-16 01:26:18] [INFO] split非依存の基本特徴量を生成中...


INFO:63_tabpfn_ensemble:split非依存の基本特徴量を生成中...


[2026-08-16 01:26:18] [INFO] ------------------------------------------------------------


INFO:63_tabpfn_ensemble:------------------------------------------------------------


[2026-08-16 01:33:01] [INFO] split非依存の基本特徴量生成完了


INFO:63_tabpfn_ensemble:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`51_`と同一・継続採用）

In [10]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")


[2026-08-16 01:33:01] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:63_tabpfn_ensemble:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-16 01:33:02] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:63_tabpfn_ensemble:入社時メモ: SVD累積寄与率=0.760


[2026-08-16 01:33:07] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:63_tabpfn_ensemble:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-16 01:33:09] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.422


INFO:63_tabpfn_ensemble:同僚からのフィードバック: SVD累積寄与率=0.422


[2026-08-16 01:33:09] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:63_tabpfn_ensemble:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`51_`と同一・継続採用）

In [11]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")


[2026-08-16 01:33:09] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:63_tabpfn_ensemble:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-16 01:35:52] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:63_tabpfn_ensemble:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（`51_`と同一）

In [12]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")


[2026-08-16 01:35:52] [INFO] Persona単位の基本特徴量を生成中...


INFO:63_tabpfn_ensemble:Persona単位の基本特徴量を生成中...


[2026-08-16 01:35:52] [INFO] Persona単位の基本特徴量処理完了


INFO:63_tabpfn_ensemble:Persona単位の基本特徴量処理完了


## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、v1=`27_`のPublic確認済み版 / v2=抽出拡張版、`51_`と同一）

`51_`と同じくv1/v2両方を生成するが、実際に特徴量として使うのはv2（`BLOCK={"L2"}`）のみ
（`28_`以降ずっとv2が現在の最良）。


In [13]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    if m:
        return m.group(1).strip()
    # 50_: 見出しがない書式B（276件、5.24%）のフォールバック（49_で確認済み・Public -0.0022〜-0.0035）。
    lines = [l for l in text.strip().splitlines() if re.search(r"勤務地|転居|在宅勤務", l)]
    return "".join(lines) if lines else None


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")


[2026-08-16 01:35:52] [INFO] 転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


INFO:63_tabpfn_ensemble:転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


[2026-08-16 01:35:52] [INFO] L_v1: Train (2761, 3), Test (2502, 3)


INFO:63_tabpfn_ensemble:L_v1: Train (2761, 3), Test (2502, 3)


[2026-08-16 01:35:52] [INFO] L_v2: Train (2761, 3), Test (2502, 3)


INFO:63_tabpfn_ensemble:L_v2: Train (2761, 3), Test (2502, 3)


## 6. 部署Target Encoding（リーク対策済）と `prepare_split` 関数（`51_`と同一）

In [14]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None, exclude_early_from_val=True):
    '''指定した分割比率で特徴量を組み立てる（51_と完全に同一ロジック）。'''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L1" in extra_blocks:
        tf = tf.merge(train_reloc_v1, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v1, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    if exclude_early_from_val and len(ag_tuning) > 0:
        n_before = len(ag_tuning)
        ag_tuning = ag_tuning[~ag_tuning.index.isin(EARLY_LEAVER_IDS)]
        logger.info(f"  検証セット: {n_before} → {len(ag_tuning)}件（早期退職者{n_before - len(ag_tuning)}名を除外）")

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了")


✅ 部署Target Encoding・prepare_split関数定義完了


## 7. 特徴量の組み立て（`51_`と同一、`BLOCK={"L2"}`固定）

In [15]:
def _feature_cols(df):
    return [c for c in df.columns if c not in ["入社日", TARGET_COL]]


BLOCK = {"L2"}   # 28_のL_v2_extended（現在の最良）に固定

logger.info("=" * 60)
logger.info("[検証用] split_80_20 / 検証=生存者のみ")
ag_train_80b, ag_val_surv, _ = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("[提出用] 全件学習（検証セットなし）")
ag_full, ag_empty, test_features_full = prepare_split(1.0, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("-" * 60)
logger.info(f"main_train={len(ag_train_80b)}, main_valid(生存者)={len(ag_val_surv)}")
logger.info(f"全件={len(ag_full)}")
logger.info(f"特徴量数: {len(_feature_cols(ag_train_80b))}")
assert len(ag_empty) == 0, "全件学習のときは検証セットが空のはず"


[2026-08-16 01:35:52] [INFO] ============================================================


INFO:63_tabpfn_ensemble:============================================================


[2026-08-16 01:35:52] [INFO] [検証用] split_80_20 / 検証=生存者のみ


INFO:63_tabpfn_ensemble:[検証用] split_80_20 / 検証=生存者のみ


[2026-08-16 01:35:53] [INFO]   検証セット: 553 → 535件（早期退職者18名を除外）


...

INFO:63_tabpfn_ensemble:main_train=2208, main_valid(生存者)=535


[2026-08-16 01:35:53] [INFO] 全件=2761


INFO:63_tabpfn_ensemble:全件=2761


[2026-08-16 01:35:53] [INFO] 特徴量数: 441


INFO:63_tabpfn_ensemble:特徴量数: 441


## 8. 特徴量グループの棚卸し（`51_`から移植、内容は同一）

In [16]:
ALL_FEATS = set(_feature_cols(ag_train_80b))

DERIVED_COLS = [
    "残業時間_mean_job_deviation", "研修時間_mean_job_deviation", "360度評価_親和度_mean_job_deviation",
    "研修時間_職種比", "研修時間_区分比",
    "初任給_等級内偏差", "初任給_区分内偏差", "月例給与_等級内偏差",
]
DEPT_TE_COLS = ["dept_target_enc", "dept_size"]


def _cols_of(df):
    return [c for c in df.columns if c != ID_COL]


_RAW_GROUPS = {
    "persona":   [c for c in train_persona.columns if c not in (ID_COL, TARGET_COL)],
    "agg":       _cols_of(train_monthly_agg),
    "catchange": _cols_of(train_cat_change),
    "missing":   _cols_of(train_missing),
    "domain":    _cols_of(train_domain),
    "advstats":  _cols_of(train_advanced_stats),
    "cluster":   _cols_of(train_cluster),
    "deptte":    DEPT_TE_COLS,
    "edafeat":   _cols_of(train_eda_feats),
    "mgr":       _cols_of(train_mgr),
    "quarterly": _cols_of(train_quarterly_exp),
    "tfidf":     [c for _df in tfidf_train_list for c in _cols_of(_df)],
    "L2":        _cols_of(train_reloc_v2),
    "derived":   DERIVED_COLS,
}

FEATURE_GROUPS = {g: [c for c in cols if c in ALL_FEATS] for g, cols in _RAW_GROUPS.items()}
ALL_GROUPS = set(FEATURE_GROUPS)

_covered = [c for cols in FEATURE_GROUPS.values() for c in cols]
_dupes = sorted({c for c in _covered if _covered.count(c) > 1})
assert not _dupes, f"複数グループに重複している列: {_dupes}"
_orphans = sorted(ALL_FEATS - set(_covered))
assert not _orphans, f"どのグループにも属さない列: {_orphans}"

print(f"特徴量 合計 {len(ALL_FEATS)} 列")
print("-" * 52)
for g in sorted(FEATURE_GROUPS, key=lambda x: -len(FEATURE_GROUPS[x])):
    print(f"  {g:<10s} {len(FEATURE_GROUPS[g]):>4d} 列   例: {FEATURE_GROUPS[g][:2]}")
print("-" * 52)
print("✅ グループ分類は全列を過不足なく覆っている")


特徴量 合計 441 列
----------------------------------------------------
  agg         224 列   例: ['残業時間_mean', '残業時間_std']
  quarterly    80 列   例: ['残業時間_q1_mean_exp', '残業時間_q2_mean_exp']
  tfidf        45 列   例: ['入社時メモ_tfidf_svd_0', '入社時メモ_tfidf_svd_1']
  advstats     25 列   例: ['残業時間_skew', '残業時間_kurtosis']
  persona      21 列   例: ['入社区分', '入社時年齢']
  catchange    14 列   例: ['部署ID_changes', '部署ID_unique_count']
  edafeat      11 列   例: ['欠勤発生月数', '欠勤_最長連続月数']
  derived       8 列   例: ['残業時間_mean_job_deviation', '研修時間_mean_job_deviation']
  missing       4 列   例: ['360度評価_親和度_missing_rate', '360度評価_信頼度_missing_rate']
  domain        3 列   例: ['engagement_score', 'overtime_stability']
  deptte        2 列   例: ['dept_target_enc', 'dept_size']
  L2            2 列   例: ['転居x勤務地_状態_v2', '転居x勤務地_ダブル悪条件_v2']
  cluster       1 列   例: ['cluster']
  mgr           1 列   例: ['初期上司_部下数']
----------------------------------------------------
✅ グループ分類は全列を過不足なく覆っている


## 9. TabPFN v2 のインストールとGPU確認

In [17]:
# tabpfn はセル1でインストール済み。ここでは何もしない。
# （もしここで再インストールすると、import済みモジュールとの不整合が起きうる）
print("tabpfn はセル1でインストール済み")


tabpfn はセル1でインストール済み


In [18]:
import torch
import tabpfn
from tabpfn import TabPFNClassifier

# バージョン確認: 2.x 系（TabPFN v2, Apache 2.0）以外だとライセンスゲートに掛かる。
# ディスク上のバージョンと、実際にimportされたバージョンを分けて見ることで、
# 「pipが効いていない」のか「再起動していないだけ」なのかを切り分ける。
from importlib.metadata import version as _pkgver
_disk = _pkgver("tabpfn")
_live = tabpfn.__version__
if not _live.startswith("2."):
    if _disk.startswith("2."):
        raise RuntimeError(
            f"ディスク上は tabpfn=={_disk}（正しい）だが、動いているカーネルには "
            f"{_live} が読み込まれたままです。\n"
            "→ **ランタイム → セッションを再起動** してから、最初のセルに戻って実行し直してください。\n"
            "   （pipで入れ替えても、import済みモジュールは再起動しないと切り替わりません）\n"
            "   再起動してもこのエラーが出る場合は「ランタイムを出荷時設定にリセット」を試してください。"
        )
    raise RuntimeError(
        f"tabpfn=={_disk} がインストールされています。v2系(2.x)が必要です。\n"
        "→ セル1の `pip install -q catboost optuna \"tabpfn==2.2.1\"` を実行し、"
        "その後セッションを再起動してください。"
    )
print(f"tabpfn version = {_live} （v2系・Apache 2.0・ライセンスゲート無し）")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device = {DEVICE}")
if DEVICE == "cpu":
    print("⚠️ GPUが無い。TabPFNはCPUだとサンプル数制限に引っかかるので、")
    print("   Colabのランタイムを「GPU」に変更してから実行し直すこと。")

# 事前登録した合格基準
REJECT_MARGIN = 0.02      # TabPFN単体valがCatBoost比 +0.02より悪ければ不合格
BLEND_WEIGHTS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]  # w=CatBoost側の重み
NOISE_FLOOR_P95 = 0.02122

A_PARAMS = {
    "depth": 4, "learning_rate": 0.03518359458951149, "l2_leaf_reg": 2.217690447016724,
    "border_count": 218, "bagging_temperature": 0.6787467566574921,
    "random_strength": 1.438494697238285,
}
ITER_FIXED = 560
SEEDS_VAL = [42, 2024, 7, 1234, 99, 555, 31337, 2718]
SEEDS_SUB = [42, 2024, 7, 1234, 99]
TABPFN_SEEDS = [42, 2024, 7]      # TabPFNもシード平均する

FEATS_ALL = _feature_cols(ag_train_80b)
print(f"特徴量 {len(FEATS_ALL)} 列 / 学習 {len(ag_train_80b)}件 / 検証(生存者) {len(ag_val_surv)}件")
assert len(FEATS_ALL) == 441, f"{len(FEATS_ALL)}列（441列のはず）"


tabpfn version = 2.2.1 （v2系・Apache 2.0・ライセンスゲート無し）
device = cuda
特徴量 441 列 / 学習 2208件 / 検証(生存者) 535件


## 10. CatBoostベースライン（同一split・同一特徴量。比較の土台）

TabPFNの良し悪しは「同じ条件のCatBoostと比べてどうか」でしか判断できないので、
このノートブック内で必ず一緒に測る。ハイパーパラメータは`54_`のA_PARAMS固定。


In [19]:
def cb_fit_predict(train_df, feats, predict_dfs, seeds, n_iter=ITER_FIXED):
    """CatBoostをシード平均で学習し、指定フレーム群への予測を返す"""
    obj = [c for c in feats if train_df[c].dtype == "object"]
    Xtr, ytr = train_df[feats].fillna(-999), train_df[TARGET_COL]
    outs = [[] for _ in predict_dfs]
    models = []
    for s in seeds:
        m = cb.CatBoostClassifier(**A_PARAMS, iterations=n_iter, random_seed=s, verbose=False,
                                  cat_features=obj, task_type="CPU")
        m.fit(Xtr, ytr)
        models.append(m)
        for k, df in enumerate(predict_dfs):
            outs[k].append(m.predict_proba(df[feats].fillna(-999))[:, 1])
    return [np.mean(o, axis=0) for o in outs], models


y_val = ag_val_surv[TARGET_COL].values
(cb_val,), cb_models = cb_fit_predict(ag_train_80b, FEATS_ALL, [ag_val_surv], SEEDS_VAL)
CB_VAL = log_loss(y_val, cb_val)
logger.info(f"CatBoost baseline val = {CB_VAL:.6f}")
print(f"CatBoost baseline (441列, 8シード平均) val logloss = {CB_VAL:.6f}")

# TabPFN用の列数感度を見るため、重要度上位150列を作る
imp = np.mean([m.get_feature_importance() for m in cb_models], axis=0)
imp_s = pd.Series(imp, index=FEATS_ALL).sort_values(ascending=False)
FEATS_TOP150 = imp_s.head(150).index.tolist()
print(f"\n重要度上位10列: {imp_s.head(10).index.tolist()}")
imp_s.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_catboost_importance.csv")

FEATURE_SETS = {"full441": FEATS_ALL, "top150": FEATS_TOP150}


[2026-08-16 01:36:36] [INFO] CatBoost baseline val = 0.506270


INFO:63_tabpfn_ensemble:CatBoost baseline val = 0.506270


CatBoost baseline (441列, 8シード平均) val logloss = 0.506270

重要度上位10列: ['転居x勤務地_状態_v2', '専攻分野', '初期職種', '転居x勤務地_ダブル悪条件_v2', '残業時間_min', '残業時間_mid_mean', '担当プロジェクト数_cv', '残業時間_q25', '上司からのフィードバック_tfidf_svd_1', '残業時間_cv']


## 11. TabPFN v2 の学習

TabPFNはスケーリング・one-hot不要でNaNもそのまま扱えるので、**CatBoostのような`fillna(-999)`はしない**
（-999で埋めると外れ値として扱われて逆効果になる）。カテゴリ列は序数コード化して
`categorical_features_indices`で位置を伝える。


In [20]:
def to_tabpfn_matrix(train_df, other_dfs, feats):
    """TabPFN用の行列を作る。NaNは埋めない。カテゴリは序数コード化して位置を返す。"""
    obj = [c for c in feats if train_df[c].dtype == "object"]
    frames = [train_df] + list(other_dfs)
    cat_map = {}
    for c in obj:
        vals = pd.concat([f[c].astype(str) for f in frames]).unique()
        cat_map[c] = {v: i for i, v in enumerate(sorted(vals))}
    out = []
    for f in frames:
        M = f[feats].copy()
        for c in obj:
            M[c] = f[c].astype(str).map(cat_map[c]).astype(float)
        out.append(M.astype(np.float32).values)
    cat_idx = [feats.index(c) for c in obj]
    return out, cat_idx


def tabpfn_fit_predict(train_df, other_dfs, feats, seeds, device=DEVICE):
    """TabPFN v2 をシード平均で学習して予測を返す"""
    (Mtr, *Mo), cat_idx = to_tabpfn_matrix(train_df, other_dfs, feats)
    ytr = train_df[TARGET_COL].values
    acc = [[] for _ in Mo]
    for s in seeds:
        kw = dict(device=device, random_state=s)
        for extra in ({"categorical_features_indices": cat_idx, "ignore_pretraining_limits": True},
                      {"categorical_features_indices": cat_idx},
                      {}):
            try:
                clf = TabPFNClassifier(**kw, **extra); break
            except TypeError:
                continue
        clf.fit(Mtr, ytr)
        for k, M in enumerate(Mo):
            acc[k].append(clf.predict_proba(M)[:, 1])
        logger.info(f"  TabPFN seed={s} 完了")
    return [np.mean(a, axis=0) for a in acc]


tp_val = {}
for name, feats in FEATURE_SETS.items():
    logger.info("=" * 60)
    logger.info(f"[TabPFN {name}] {len(feats)}列で学習開始")
    import time as _t; _t0 = _t.time()
    (pv,) = tabpfn_fit_predict(ag_train_80b, [ag_val_surv], feats, TABPFN_SEEDS)
    tp_val[name] = pv
    ll = log_loss(y_val, pv)
    print(f"[TabPFN {name}] {len(feats):3d}列  val logloss = {ll:.6f}  "
          f"(CatBoost比 {ll - CB_VAL:+.6f})  所要 {(_t.time()-_t0)/60:.1f}分")
    np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_tabpfn_{name}_valpreds.npy", pv)


[2026-08-16 01:36:36] [INFO] ============================================================


INFO:63_tabpfn_ensemble:============================================================


[2026-08-16 01:36:36] [INFO] [TabPFN full441] 441列で学習開始


INFO:63_tabpfn_ensemble:[TabPFN full441] 441列で学習開始


tabpfn-v2-classifier-finetuned-zk73skhh.(…):   0%|          | 0.00/29.0M [00:00<?, ?B/s]

...

[2026-08-16 01:39:56] [INFO]   TabPFN seed=2024 完了


INFO:63_tabpfn_ensemble:  TabPFN seed=2024 完了


[2026-08-16 01:40:14] [INFO]   TabPFN seed=7 完了


INFO:63_tabpfn_ensemble:  TabPFN seed=7 完了


[TabPFN top150] 150列  val logloss = 0.507569  (CatBoost比 +0.001299)  所要 0.9分


## 12. 合格判定・多様性・ブレンド重み走査

In [21]:
print(f"CatBoost baseline val = {CB_VAL:.6f}")
print(f"足切り基準: TabPFN単体val > {CB_VAL + REJECT_MARGIN:.6f} なら不合格（事前登録）")
print("=" * 78)

rows = []
for name, pv in tp_val.items():
    ll = log_loss(y_val, pv)
    corr = float(np.corrcoef(cb_val, pv)[0, 1])
    ok = ll <= CB_VAL + REJECT_MARGIN
    best_ll, best_w = min((log_loss(y_val, np.clip(w * cb_val + (1 - w) * pv, 1e-9, 1 - 1e-9)), w)
                          for w in BLEND_WEIGHTS)
    rows.append({"構成": name, "列数": len(FEATURE_SETS[name]), "TabPFN単体val": ll,
                 "CatBoost比": ll - CB_VAL, "CatBoostとの相関": corr,
                 "最良ブレンド": best_ll, "その時のw_CatBoost": best_w,
                 "ブレンド改善": best_ll - CB_VAL, "合格": "○" if ok else "×（足切り）"})
res = pd.DataFrame(rows)
pd.set_option("display.width", 240)
print(res.round(6).to_string(index=False))

print()
for name, pv in tp_val.items():
    print(f"--- [{name}] ブレンド重み走査（w=CatBoost側の重み）---")
    for w in BLEND_WEIGHTS:
        ll = log_loss(y_val, np.clip(w * cb_val + (1 - w) * pv, 1e-9, 1 - 1e-9))
        mark = ""
        if w == 1.0: mark = "  <- CatBoost単体"
        elif w == 0.0: mark = "  <- TabPFN単体"
        elif ll < CB_VAL: mark = "  ★ CatBoost単体より良い"
        print(f"   w={w:.1f}: {ll:.6f}{mark}")

res.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_blend_scan.csv", index=False)
PASS = res[res["合格"] == "○"]
print()
if len(PASS) == 0:
    print("=" * 78)
    print("全構成が足切り。55_のGRUと同じパターン（多様性はあっても絶対性能の差が大きすぎる）。")
    print("この場合は提出せず、負の結果として記録して終了する。")
else:
    print("=" * 78)
    print(f"合格 {len(PASS)}構成。次セルで提出ファイルを作る。")
    print("※ ただし検証の『改善』は根拠にならない（[[validation-asymmetry]]）。採否はPublicのみ。")


CatBoost baseline val = 0.506270
足切り基準: TabPFN単体val > 0.526270 なら不合格（事前登録）
     構成  列数  TabPFN単体val  CatBoost比  CatBoostとの相関   最良ブレンド  その時のw_CatBoost    ブレンド改善 合格
full441 441     0.522101   0.015831      0.943999 0.506045             0.9 -0.000224  ○
 top150 150     0.507569   0.001299      0.954246 0.502252             0.5 -0.004018  ○

--- [full441] ブレンド重み走査（w=CatBoost側の重み）---
   w=0.0: 0.522101  <- TabPFN単体
   w=0.1: 0.518310
   w=0.2: 0.515161
   w=0.3: 0.512562
   w=0.4: 0.510451
   w=0.5: 0.508786
   w=0.6: 0.507533
   w=0.7: 0.506668
   w=0.8: 0.506176  ★ CatBoost単体より良い
   w=0.9: 0.506045  ★ CatBoost単体より良い
   w=1.0: 0.506270  <- CatBoost単体
--- [top150] ブレンド重み走査（w=CatBoost側の重み）---
   w=0.0: 0.507569  <- TabPFN単体
   w=0.1: 0.505590  ★ CatBoost単体より良い
   w=0.2: 0.504122  ★ CatBoost単体より良い
   w=0.3: 0.503102  ★ CatBoost単体より良い
   w=0.4: 0.502488  ★ CatBoost単体より良い
   w=0.5: 0.502252  ★ CatBoost単体より良い
   w=0.6: 0.502373  ★ CatBoost単体より良い
   w=0.7: 0.502838  ★ CatBoost単体より良い
   w=0.8: 0.5

## 13. 提出ファイルの作成（合格構成のみ）

Train全件で再学習し、Testを予測する。ブレンド重みは**検証で選んだ値をそのまま使わず**、
0.5固定と検証最良の2つを出す（検証で選んだ重みは[[ensemble-oof-overfitting]]の
「重み学習は過学習する」に該当しうるため、素朴な0.5も並べて両方Publicで確かめる）。


In [22]:
def save_submission(idx, preds, label):
    path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{label}.csv"
    pd.DataFrame({ID_COL: idx, TARGET_COL: preds}).to_csv(path, index=False, header=False)
    logger.info(f"  提出ファイル: {path.name}（予測平均={preds.mean():.4f}）")
    return str(path)


_best = sorted((PROJECT_ROOT / "data" / "output").glob(
    "*/*_50_autogluon_memofix_AG50_full441_weighted.csv"))
BEST_PRED = (pd.read_csv(_best[-1], header=None, names=[ID_COL, "pred"]).set_index(ID_COL)["pred"]
             if _best else None)

subs = []
if len(PASS) > 0:
    for _, r in PASS.iterrows():
        name = r["構成"]; feats = FEATURE_SETS[name]
        logger.info(f"[{name}] 全件学習して提出ファイルを作成")
        (cb_te,), _ = cb_fit_predict(ag_full, feats, [test_features_full], SEEDS_SUB)
        (tp_te,) = tabpfn_fit_predict(ag_full, [test_features_full], feats, TABPFN_SEEDS)
        np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_tabpfn_{name}_testpreds.npy", tp_te)
        for tag, w in [("w050", 0.5), ("wbest", float(r["その時のw_CatBoost"]))]:
            if w in (0.0, 1.0) and tag == "wbest":
                print(f"  [{name}] 検証最良の重みが w={w}（単体）なのでブレンド提出はしない")
                continue
            pr = w * cb_te + (1 - w) * tp_te
            label = f"TP63_{name}_blend_{tag}"
            path = save_submission(test_features_full.index, pr, label)
            row = {"config": label, "w_catboost": w, "pred_mean": float(pr.mean()),
                   "submission_path": path}
            if BEST_PRED is not None:
                al = pd.Series(pr, index=test_features_full.index).loc[BEST_PRED.index]
                row["mad_vs_best"] = float(np.abs(al.values - BEST_PRED.values).mean())
                row["corr_vs_best"] = float(np.corrcoef(al.values, BEST_PRED.values)[0, 1])
            subs.append(row)

if subs:
    S = pd.DataFrame(subs)
    S["提出"] = np.where(S.get("mad_vs_best", 1) > NOISE_FLOOR_P95, "提出する",
                          "見送り（現最良との差がノイズ床以下）")
    print(S.round(6).to_string(index=False))
    S.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_submissions.csv", index=False)
    S.to_csv(CHECKPOINT_PATH, index=False)
else:
    print("提出ファイルなし（全構成が足切り、または合格構成でブレンドが単体に負けた）。")
logger.info("完了")


[2026-08-16 01:40:14] [INFO] [full441] 全件学習して提出ファイルを作成


INFO:63_tabpfn_ensemble:[full441] 全件学習して提出ファイルを作成


[2026-08-16 01:42:49] [INFO]   TabPFN seed=42 完了


INFO:63_tabpfn_ensemble:  TabPFN seed=42 完了


[2026-08-16 01:45:01] [INFO]   TabPFN seed=2024 完了


...

INFO:63_tabpfn_ensemble:  提出ファイル: 20260816_63_tabpfn_ensemble_TP63_top150_blend_w050.csv（予測平均=0.5933）


[2026-08-16 01:49:22] [INFO]   提出ファイル: 20260816_63_tabpfn_ensemble_TP63_top150_blend_wbest.csv（予測平均=0.5933）


INFO:63_tabpfn_ensemble:  提出ファイル: 20260816_63_tabpfn_ensemble_TP63_top150_blend_wbest.csv（予測平均=0.5933）


                  config  w_catboost  pred_mean                                                                                                  submission_path  mad_vs_best  corr_vs_best   提出
 TP63_full441_blend_w050         0.5   0.597108  /content/drive/MyDrive/jaggle_2026/data/output/20260816/20260816_63_tabpfn_ensemble_TP63_full441_blend_w050.csv     0.047386      0.974578 提出する
TP63_full441_blend_wbest         0.9   0.591572 /content/drive/MyDrive/jaggle_2026/data/output/20260816/20260816_63_tabpfn_ensemble_TP63_full441_blend_wbest.csv     0.047497      0.974749 提出する
  TP63_top150_blend_w050         0.5   0.593327   /content/drive/MyDrive/jaggle_2026/data/output/20260816/20260816_63_tabpfn_ensemble_TP63_top150_blend_w050.csv     0.044995      0.977616 提出する
 TP63_top150_blend_wbest         0.5   0.593327  /content/drive/MyDrive/jaggle_2026/data/output/20260816/20260816_63_tabpfn_ensemble_TP63_top150_blend_wbest.csv     0.044995      0.977616 提出する
[2026-08-16 01:49:22] [INFO] 完了


INFO:63_tabpfn_ensemble:完了


## 14. 提出方針

- **採否はPublicのみ**（[[validation-asymmetry]]）。検証は足切り専用。
- 足切りは事前登録済み: TabPFN単体valがCatBoost比 **+0.02より悪ければ不合格**。
  `55_`のGRU（多様性はあったが絶対差0.136で全滅）の再現を避けるため。
- 提出は `w=0.5`（素朴な固定重み）と `w=検証最良` の2本。検証で選んだ重みだけを信じるのは
  [[ensemble-oof-overfitting]]の「重み学習は過学習する」に該当しうるので、両方をPublicで確かめる。

### どちらに転んでも得られるもの

| 結果 | 意味 |
|---|---|
| TabPFNがCatBoostに肉薄し、ブレンドが改善 | GBDT以外の帰納バイアスが効く。AutoGluonへの組み込みへ進む |
| 単体は近いがブレンドが改善しない | 多様性が見かけだけ（予測相関が高い）と分かる |
| 単体が足切り | 表形式基盤モデルもこのデータでは歯が立たない。[[monthly-data-information-ceiling]]の裏付けが増える |

### ライセンス

TabPFN **v2** は Apache 2.0（帰属表示が追加要件）。新しい 2.5/2.6/3 は非商用ライセンスなので、
本ノートブックでは意図的に v2 を使っている。バージョンを上げる場合はライセンスを再確認すること。
